# 01 — Getting Started with `isa_phm`

This notebook introduces the `ISAWrapper` API and shows how to:

1. Load an ISA-PHM JSON file
2. Inspect the investigation hierarchy (studies → assays → runs)
3. Explore sensors, protocols, and factor structure
4. Check what data files are linked and whether they exist on disk

No measurement data files are required for this notebook — all operations here work purely from the ISA-JSON metadata.

---
**Dataset used**: `notebooks/single-run-isa.json` (diagnostic, 2 studies, 6 sensors each)

In [12]:
# Install package dependencies into the active kernel (run once)
%pip install -q pydantic pandas numpy scipy matplotlib --quiet

Note: you may need to restart the kernel to use updated packages.


In [13]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys
import logging
from pathlib import Path

# Add the python-wrapper directory to sys.path if running from notebooks/ sub-folder.
ROOT = Path("..").resolve()        # python-wrapper/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Suppress noisy duplicate-@id warnings from shared ontology annotations.
logging.getLogger("isa_phm").setLevel(logging.ERROR)

# Paths — assumes this notebook lives at python-wrapper/notebooks/
ISA_JSON  = Path(r"E:\\XJTU-SY_Bearing_Datasets\\XJTU-SY Bearing Datasets-ISA-PHM-2.json").resolve()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("Exists   :", ISA_JSON.exists())

ISA-JSON : E:\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM-2.json
Exists   : True


## 1. Load the ISA-JSON file

`ISAWrapper` chains the four core components automatically:
- **ISAParser** — reads and optionally validates the JSON
- **ISAPreprocessor** — applies the 5 auto-fix rules (normalise paths, strip whitespace, etc.)
- **MetadataExtractor** — builds typed Pydantic domain models
- **DataIntegrator** — sets up CSV loading + FIFO cache for later

`strict_validation=False` skips `isatools` schema validation (useful when `isatools` is not installed).

In [14]:
from isa_phm import ISAWrapper

wrapper = ISAWrapper(
    path=ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,   # set True if isatools is installed
    auto_fix=True,
)
print(wrapper)

ISAWrapper(title='XJTU-SY Bearing Datasets', n_studies=15, experiment_type='prognostics-experiment')


## 2. Investigation overview

The top-level summary: title, experiment type, number of studies and contacts.

In [15]:
ov = wrapper.investigation_overview()
print(f"Title          : {ov.title}")
print(f"Identifier     : {ov.identifier}")
print(f"Experiment type: {ov.experiment_type}")
print(f"Studies        : {ov.n_studies}")
print(f"Contacts       : {ov.n_contacts}")
print(f"Description    : {ov.description[:120]}...")

Title          : XJTU-SY Bearing Datasets
Identifier     : e41c5c45-b0c4-4c3c-808f-bf2f75a6302e
Experiment type: prognostics-experiment
Studies        : 15
Contacts       : 4
Description    : XJTU-SY bearing datasets are provided by the Institute of Design Science and Basic Component at Xi’an Jiaotong Universit...


## 3. List studies

`list_studies()` returns one `StudySummary` per study — no file I/O.

In [16]:
import pandas as pd

studies = wrapper.list_studies()
pd.DataFrame([s.model_dump() for s in studies])

,study_id,title,n_assays,n_runs,n_factors
0,12a95728-7b62-4f7f-8c49-82b4512fe5d2,Bearing 1_1,2,123,5
1,f8dd58d5-e816-4da4-9b76-f4c12548912c,Bearing 1_2,2,161,5
2,e5e0a339-5511-419f-9468-d7a9ba4c9efd,Bearing 1_3,2,158,5
3,6b91085b-abf5-4390-932f-7b296e33be78,Bearing 1_4,2,122,5
4,aab88984-26cb-416f-82f3-9d6a008cc9b5,Bearing 1_5,2,52,5
5,bffdb9f9-e7ed-4cc4-a04d-d340b7f3f83f,Bearing 2_1,2,491,5
6,88cb8bb9-500e-4f78-a765-153367788033,Bearing 2_2,2,161,5
7,36a08c35-c335-4099-91b7-c1a4aaa127db,Bearing 2_3,2,533,5
8,3321beba-8621-4147-92a8-0f373940c147,Bearing 2_4,2,42,5
9,7912ecac-e06c-42dc-81d6-4ece49f19104,Bearing 2_5,2,339,5


## 4. Drill into a study

Navigate with `wrapper.study("title or UUID")`. The lookup is case-insensitive on the title.

In [17]:
# Pick the first study by title
first_title = studies[0].title
study = wrapper.study(first_title)
print(f"Navigated to : {study.title!r}")
print(f"Run count    : {study.run_count}")
print(f"Has runs     : {study.has_runs}   ← False = diagnostic (1 run per assay)")

# Structured overview
study_ov = study.overview()
print(f"\nFactors ({len(study_ov.factors)}):")
for f in study_ov.factors:
    unit = f"[{f.unit}]" if f.unit else ""
    print(f"  {f.factor_name} {unit} — {f.factor_type}")

Navigated to : 'Bearing 1_1'
Run count    : 123
Has runs     : True   ← False = diagnostic (1 run per assay)

Factors (5):
  Fault Type  — Qualitative fault specification
  Bearing Lifetime [min] — Quantitative fault specification
  Motor speed [RPM] — Operating condition
  Pressure Axial [kN] — Operating condition
  Pressure Radial [kN] — Operating condition


## 5. List assays for a study

Each assay corresponds to one sensor. `list_assays()` returns a list of `AssaySummary` objects.

In [20]:
assay_summaries = study.list_assays()
pd.DataFrame([a.model_dump() for a in assay_summaries])

,assay_id,sensor_id,sensor_alias,technology_type,measurement_type,n_runs,n_processed_files
0,a_st01_se01,be71e503-a482-4655-9d5c-6217acca2c99,Accel_Axial,Accelerometer,Vibration,123,0
1,a_st01_se02,3638a9b3-a3f8-402e-ab22-1966897e2ec0,Accel_Radial,Accelerometer,Vibration,123,0


## 6. Inspect experimental factors

`list_factors()` returns the controlled variables (fault type, severity, speed …).

In [19]:
factors = study.list_factors()
pd.DataFrame([f.model_dump() for f in factors])

,factor_id,factor_name,factor_type,unit
0,#study_factor/b8c61e80-2e1c-4158-8f2f-54fd9323...,Fault Type,Qualitative fault specification,None
1,#study_factor/94599c85-ee80-48f8-afca-84165b7c...,Bearing Lifetime,Quantitative fault specification,min
2,#study_factor/5b82f193-a675-4661-8721-6f53daf6...,Motor speed,Operating condition,RPM
3,#study_factor/f4dc0488-35aa-4b0f-9070-f0b104a5...,Pressure Axial,Operating condition,kN
4,#study_factor/f1a030b1-f6c4-4881-beb9-f97172ba...,Pressure Radial,Operating condition,kN


## 7. Navigate into an assay

An assay is the link between a study and a sensor. Use `study.assay(assay_id)` to get an `AssayProxy`.

In [21]:
# Use the first assay_id from the summary list
first_assay_id = assay_summaries[0].assay_id
assay = study.assay(first_assay_id)

assay_ov = assay.overview()
print(f"Assay ID          : {assay_ov.assay_id}")
print(f"Sensor alias      : {assay_ov.sensor_alias}")
print(f"Measurement type  : {assay_ov.measurement_type}")
print(f"Technology type   : {assay_ov.technology_type}")
print(f"Runs              : {assay_ov.n_runs}")
print(f"\nRun IDs: {[r.run_id for r in assay_ov.runs]}")

Assay ID          : a_st01_se01
Sensor alias      : Accel_Axial
Measurement type  : Vibration
Technology type   : Accelerometer
Runs              : 123

Run IDs: ['run_001', 'run_002', 'run_003', 'run_004', 'run_005', 'run_006', 'run_007', 'run_008', 'run_009', 'run_010', 'run_011', 'run_012', 'run_013', 'run_014', 'run_015', 'run_016', 'run_017', 'run_018', 'run_019', 'run_020', 'run_021', 'run_022', 'run_023', 'run_024', 'run_025', 'run_026', 'run_027', 'run_028', 'run_029', 'run_030', 'run_031', 'run_032', 'run_033', 'run_034', 'run_035', 'run_036', 'run_037', 'run_038', 'run_039', 'run_040', 'run_041', 'run_042', 'run_043', 'run_044', 'run_045', 'run_046', 'run_047', 'run_048', 'run_049', 'run_050', 'run_051', 'run_052', 'run_053', 'run_054', 'run_055', 'run_056', 'run_057', 'run_058', 'run_059', 'run_060', 'run_061', 'run_062', 'run_063', 'run_064', 'run_065', 'run_066', 'run_067', 'run_068', 'run_069', 'run_070', 'run_071', 'run_072', 'run_073', 'run_074', 'run_075', 'run_076', '

In [22]:
# Standard DataFrame columns this assay will produce when data files are loaded
print("Standard columns:", assay.columns())

# list_runs() returns one RunSummary per run (1 for diagnostic, N for prognostic)
run_summaries = assay.list_runs()
print(f"\nRun count: {len(run_summaries)}")
for r in run_summaries:
    print(f"  {r.run_id}  raw={r.raw_file_path}  processed={r.processed_file_path}")

Standard columns: ['time', 'value', 'study_id', 'assay_id', 'run_id', 'sensor_alias', 'measurement_type', 'file_type']

Run count: 123
  run_001  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\1_horizontal.csv  processed=None
  run_002  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\2_horizontal.csv  processed=None
  run_003  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\3_horizontal.csv  processed=None
  run_004  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\4_horizontal.csv  processed=None
  run_005  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\5_horizontal.csv  processed=None
  run_006  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\6_horizontal.csv  processed=None
  run_007  raw=E:\XJTU-SY_Bearing_Datasets\Data_split_per_file\35Hz12kN\Bearing1_1\7_horizontal.csv  processed=None
  run_008  raw=E:\XJTU-SY_Bearing_Datasets\Data_split

## 8. Loading data (graceful degradation)

The fixture CSV paths point to an external drive that may not be present on your machine.  
The wrapper surfaces a `DataFileError` instead of crashing so you can handle it explicitly.

In [27]:
from isa_phm.errors import DataFileError

# For multi-run assays, a run_id must be supplied explicitly.
# Use the first run if there are multiple; None lets the wrapper auto-select for single-run assays.
run_id = run_summaries[0].run_id if len(run_summaries) > 1 else None

# Prefer processed files; fall back to raw if no processed path is recorded.
first_run = run_summaries[0]
file_type = "processed" if first_run.processed_file_path is not None else "raw"
print(f"Loading file_type={file_type!r}, run_id={run_id!r}")

try:
    df = assay.load_dataframe(run_id=run_id, file_type=file_type)
    print(f"Loaded {len(df)} rows × {len(df.columns)} cols")
    display(df.head())
except DataFileError as exc:
    print(f"[DataFileError] {exc}")
    print("(CSV file not found — expected for demo fixtures with external drive paths)")


Loading file_type='raw', run_id='run_001'
Loaded 32768 rows × 8 cols


,time,value,study_id,assay_id,run_id,sensor_alias,measurement_type,file_type
0,0.000000,-0.396395,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,run_001,Accel_Axial,Vibration,raw
1,0.000039,-0.123107,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,run_001,Accel_Axial,Vibration,raw
2,0.000078,0.988841,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,run_001,Accel_Axial,Vibration,raw
3,0.000117,0.006676,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,run_001,Accel_Axial,Vibration,raw
4,0.000156,-1.074386,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,run_001,Accel_Axial,Vibration,raw


## 9. Repair log

During loading the preprocessor may auto-fix issues (path normalisation, missing fields, …).  
`wrapper.repair_log` exposes a structured, human-readable summary.

In [28]:
log = wrapper.repair_log
print(f"Total repairs : {len(log)}")
print(f"Warnings      : {len(log.warnings())}")
print()

if len(log) == 0:
    print("No repairs applied.")
else:
    # Print at most 10 repairs to keep output readable
    for action in log.actions[:10]:
        print(action)
    if len(log) > 10:
        print(f"... and {len(log) - 10} more")

Total repairs : 18432
Warnings      : 0

[INFO] DataFile(#data_file/d7ffb0b4-9e8c-4b3e-aaa5-fdc411081fa4).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/c3f8f6d4-d89a-40f7-b68b-211df22b58be).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/d891ab6d-8176-4a40-9c20-c1fb729debc0).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/91ec6ec4-53e9-48bc-ac23-82aad9c1a757).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/97cc8872-0860-4066-bb5c-b3b75d880952).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/2870c015-d455-4829-9ae9-87eebca738e4).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/a5b3ddb8-e2b4-4861-97ae-fdd19e71af03).name: Relative/Windows path resolved against data_root.
[INFO] DataFile(#data_file/5f4cd336-9452-41e6-b6e9-57fbb3d8546e).name: Relative/Windows path resolved against da

---

## What's next?

| Notebook | Topic |
|---|---|
| `02_diagnostic_workflow.ipynb` | Load signal data, frequency-domain plots, missing-value report |
| `03_prognostic_workflow.ipynb` | Multi-run lifecycle features, trend plots, correlation analysis |

The API reference lives in `python-wrapper/isa_phm/` — each module is a single focused class.